In [1]:
import pdfplumber
import pandas as pd
from tqdm import tqdm

In [2]:
file = pdfplumber.open("SquadLists-English.pdf")

In [ ]:
tables = []
for page in tqdm(file.pages):
    table = page.extract_table()
    table_df = pd.DataFrame(table[1:], columns=table[0])
    table_df.columns = [str(col) for col in table_df.columns]
    table_df.drop(labels=['nan'], axis=1, inplace=True)
    table_df.dropna(inplace=True)
    country = page.extract_text_lines()[3]['text']
    name, short = country[:-1].split('(')
    table_df['NATIONALITY']  = name.strip()
    table_df['NATIONALITY SHORT'] = short.strip()
    tables.append(table_df)

  0%|          | 0/48 [00:00<?, ?it/s]Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
100%|██████████| 48/48 [00:32<00:00,  1.47it/s]


In [85]:
df = pd.concat(tables).reset_index(drop=True)

In [86]:
df['CLUBS COUNTRY SHORT'] = df['CLUB'].apply(lambda x: x.split('(')[1].strip(')'))
df['CLUB'] = df['CLUB'].apply(lambda x: x.split('(')[0].strip())

In [87]:
# df['CLUB'] = df['CLUB'].apply(lambda x: 'SL Benfica' if 'SL Ben' in x else x)
# df['CLUB'] = df['CLUB'].apply(lambda x: 'Sheffield United FC' if ('Shef' in x and 'United FC' in x) else x)
# df['CLUB'] = df['CLUB'].apply(lambda x: 'AE Kifisia FC' if ('AE Ki' in x and 'sia FC' in x) else x)
# df['CLUB'] = df['CLUB'].apply(lambda x: 'CA Vélez Sarsfield' if ('CA Vélez Sars' in x and 'eld' in x) else x)

In [88]:
for col in ['PLAYER NAME', 'FIRST NAME(S)', 'CLUB']:
    df[col] = df[col].str.replace('\x00', 'fi')

In [89]:
df['IN-SQUAD ID'] = df['#'].copy()
df.drop(columns=['#'], inplace=True)

In [90]:
df.columns = ['POSITION', 'PLAYER_NAME', 'FIRST_NAME', 'LAST_NAME', 'NAME_ON_SHIRT',
       'DATE_OF_BIRTH', 'CLUB', 'HEIGHT', 'CAPS', 'GOALS', 'NATIONALITY',
       'NATIONALITY_SHORT', 'CLUB_COUNTRY_SHORT', 'IN_SQUAD_ID']

In [91]:
df = df[['IN_SQUAD_ID', 'POSITION', 'PLAYER_NAME', 'FIRST_NAME', 'LAST_NAME', 'NAME_ON_SHIRT',
       'NATIONALITY', 'NATIONALITY_SHORT', 'CLUB', 'CLUB_COUNTRY_SHORT',
       'DATE_OF_BIRTH', 'HEIGHT', 'CAPS', 'GOALS']]

In [94]:
df.to_csv("players_wc2026.csv", index=False)